# Chapter 21 — Regression for Engineers
*Track 4: Engineers*

## 🎯 Learning Objectives
- Apply linear and polynomial regression to engineering data
- Check OLS assumptions (residual diagnostics)
- Use regression for calibration, degradation, and load prediction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd
import statsmodels.api as sm
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — OLS Regression

$$Y = \beta_0 + \beta_1 X + \varepsilon, \quad \varepsilon \sim N(0, \sigma^2)$$

OLS estimators:
$$\hat\beta = (X^T X)^{-1} X^T y$$

**Gauss-Markov theorem**: OLS is BLUE (Best Linear Unbiased Estimator)
under: linearity, independence, homoscedasticity, zero-mean errors.

In [ ]:
# Sensor calibration: true vs measured temperature
T_true = rng.uniform(0, 100, 60)
T_meas = 1.05 * T_true - 2.3 + rng.normal(0, 1.5, 60)  # gain error + offset + noise

X_mat = sm.add_constant(T_meas)
model = sm.OLS(T_true, X_mat).fit()
print(model.summary().tables[1])

plt.figure(figsize=(8, 5))
plt.scatter(T_meas, T_true, alpha=0.6, s=30)
x_range = np.linspace(T_meas.min(), T_meas.max(), 100)
plt.plot(x_range, model.predict(sm.add_constant(x_range)), "r-", lw=2, label="Calibration curve")
plt.plot([0,110], [0,110], "k--", alpha=0.5, label="Ideal (y=x)")
plt.xlabel("Measured Temperature"); plt.ylabel("True Temperature")
plt.title("Sensor Calibration Regression")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Residual Diagnostics

For OLS to be valid, check:
1. **Residuals vs. fitted**: should show no pattern (homoscedasticity)
2. **QQ plot**: residuals should be normal
3. **Scale-location**: sqrt(|residuals|) should be flat
4. **Leverage**: influential points should not dominate

In [ ]:
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# 1. Residuals vs fitted
axes[0,0].scatter(fitted, residuals, alpha=0.5, s=20)
axes[0,0].axhline(0, color="red"); axes[0,0].set_title("Residuals vs Fitted")
axes[0,0].set_xlabel("Fitted"); axes[0,0].set_ylabel("Residuals")

# 2. QQ plot
stats.probplot(residuals, plot=axes[0,1])
axes[0,1].set_title("Q-Q Plot of Residuals")

# 3. Scale-location
axes[1,0].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.5, s=20)
axes[1,0].set_title("Scale-Location"); axes[1,0].set_xlabel("Fitted")
axes[1,0].set_ylabel("√|Residuals|")

# 4. Residuals histogram
axes[1,1].hist(residuals, bins=20, density=True, alpha=0.7)
xr = np.linspace(residuals.min(), residuals.max(), 100)
axes[1,1].plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()), "r-")
axes[1,1].set_title("Residuals Histogram")
plt.tight_layout(); plt.show()

sw_stat, sw_p = stats.shapiro(residuals)
print(f"Shapiro-Wilk normality test: W={sw_stat:.4f}, p={sw_p:.4f}")

In [ ]:
# Degradation modelling: component wear over time
time_hours = np.arange(0, 1001, 50, dtype=float)
wear = 0.0005 * time_hours + 0.000002 * time_hours**2 + rng.normal(0, 0.02, len(time_hours))

# Polynomial regression
poly = make_pipeline(PolynomialFeatures(2), LinearRegression())
poly.fit(time_hours.reshape(-1,1), wear)

t_pred = np.linspace(0, 1500, 300)
wear_pred = poly.predict(t_pred.reshape(-1,1))
failure_threshold = 0.8
t_failure_idx = np.searchsorted(wear_pred, failure_threshold)
t_failure = t_pred[min(t_failure_idx, len(t_pred)-1)]

plt.figure(figsize=(9, 5))
plt.scatter(time_hours, wear, alpha=0.7, s=20, label="Measured wear")
plt.plot(t_pred, wear_pred, "r-", lw=2, label="Regression model")
plt.axhline(failure_threshold, color="black", linestyle="--", label=f"Failure threshold={failure_threshold}")
if t_failure < t_pred[-1]:
    plt.axvline(t_failure, color="orange", linestyle="--", label=f"Predicted failure: {t_failure:.0f}h")
plt.xlabel("Operating hours"); plt.ylabel("Wear [mm]")
plt.title("Component Degradation — Polynomial Regression")
plt.legend(); plt.tight_layout(); plt.show()